In [ ]:
# CONFIG
import os
import glob

def env_flag(name, default=True):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() not in {'0', 'false', 'no', 'off'}


SEED = 42
EVAL_METHOD = os.environ.get('EVAL_METHOD', 'split').strip().lower()
if EVAL_METHOD in {'cv', 'kfold', 'k-fold'}:
    EVAL_METHOD = 'crossfit'
if EVAL_METHOD not in {'crossfit', 'split'}:
    raise ValueError("EVAL_METHOD must be 'crossfit' or 'split'")
N_SPLITS = int(os.environ.get('N_SPLITS', '5'))
TEST_SIZE = float(os.environ.get('TEST_SIZE', '0.30'))
N_BOOT = 500
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95)
CORR_THRESHOLD = 0.70

# Runtime switches. Override on the cluster, e.g. RUN_BART_SLEARNER=0 for a smoke run.
RUN_XGB_SLEARNER = env_flag('RUN_XGB_SLEARNER', True)
RUN_LASSO_SLEARNER = env_flag('RUN_LASSO_SLEARNER', True)
RUN_NEURAL_SLEARNER = env_flag('RUN_NEURAL_SLEARNER', True)
RUN_BART_SLEARNER = env_flag('RUN_BART_SLEARNER', True)
RUN_CAUSAL_FOREST = env_flag('RUN_CAUSAL_FOREST', True)
USE_TF_GPU = env_flag('USE_TF_GPU', False)
NEURAL_BACKEND = os.environ.get('NEURAL_BACKEND', 'keras' if USE_TF_GPU else 'sklearn').lower()

# BART settings mirror the submitted notebooks. Increase draws for final archival runs if time allows.
BART_DRAW_SCALE = float(os.environ.get('BART_DRAW_SCALE', '1.0'))
BART_DRAWS = {
    'eICU': int(50 * BART_DRAW_SCALE),
    'PMAP': int(50 * BART_DRAW_SCALE),
    'MIMIC-IV': int(50 * BART_DRAW_SCALE),
    'HYPERION': int(100 * BART_DRAW_SCALE),
}
BART_TUNE = int(float(os.environ.get('BART_TUNE_SCALE', '1.0')) * 50)
BART_CHAINS = 2
BART_CORES = 1

REPO_ROOT = os.path.abspath('..')
OUTDIR = os.path.abspath('reviewer_robust_outputs')
os.makedirs(OUTDIR, exist_ok=True)

CSV_CANDIDATES = {
    'eICU': [
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictorsDiag.csv'),
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictors*.csv'),
    ],
    'PMAP': [
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_Predictors2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_*2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_Predictors4.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP*Predictors*.csv'),
    ],
    'MIMIC-IV': [
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_*Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_Predictors1.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC*Predictors*.csv'),
    ],
    'HYPERION': [
        os.path.join(REPO_ROOT, 'hyperion', 'predictorsDf.csv'),
        os.path.join(REPO_ROOT, 'hyperion', '*predictors*.csv'),
    ],
}

def resolve_csv(name):
    matches = []
    for candidate in CSV_CANDIDATES[name]:
        hits = sorted(glob.glob(candidate))
        if hits:
            matches.extend(hits)
    if not matches:
        return CSV_CANDIDATES[name][0]
    # Preserve candidate priority while de-duplicating.
    seen = set()
    ordered = []
    for p in matches:
        if p not in seen:
            ordered.append(p)
            seen.add(p)
    return ordered[0]

CSV = {name: resolve_csv(name) for name in CSV_CANDIDATES}
print('Resolved predictor CSVs:')
for name, path in CSV.items():
    print(f'  {name}: {path}')

# TensorFlow on the cluster can fail if it sees a GPU but CUDA compiler tools
# (ptxas/nvlink) are absent. Default neural nets to CPU; opt in with USE_TF_GPU=1.
if not USE_TF_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
    os.environ['XLA_FLAGS'] = ''

DATASETS = [
    {'name': 'eICU',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'PMAP',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'MIMIC-IV', 'observational': True,  'treatment_col': 'TTM'},
    {'name': 'HYPERION', 'observational': False, 'treatment_col': 'TTM'},
]
OUTCOMES = {
    'mortality': 'mortality',
    'neuro': 'neuro_favorable',
}


In [ ]:
import os
import ast
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

try:
    from econml.dml import CausalForestDML
except Exception as e:
    CausalForestDML = None
    print('econml unavailable:', e)

if RUN_NEURAL_SLEARNER and NEURAL_BACKEND == 'keras':
    try:
        import tensorflow as tf
        if not USE_TF_GPU:
            try:
                tf.config.set_visible_devices([], 'GPU')
                tf.config.optimizer.set_jit(False)
            except Exception as _tf_gpu_e:
                print('TensorFlow GPU disable warning:', _tf_gpu_e)
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception as e:
        tf = None
        keras = None
        layers = None
        print('TensorFlow unavailable; neural model will fall back to sklearn MLP:', e)
        NEURAL_BACKEND = 'sklearn'
else:
    tf = None
    keras = None
    layers = None

try:
    import pymc as pm
    import pymc_bart as pmb
except Exception as e:
    pm = None
    pmb = None
    print('PyMC/BART unavailable:', e)

np.random.seed(SEED)
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 80)


In [ ]:
CURATED = {
 'eICU': ['gender','age','bmi',
    'nurse_first_Non-Invasive BP Systolic','nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Mean','nurse_first_Heart Rate','nurse_first_O2 Saturation',
    'lab_first_Respiratory Rate','lab_first_FiO2','nurse_first_GCS Total','nurse_first_Motor',
    'nurse_first_QTc','lab_first_pH','lab_first_paO2','lab_first_paCO2','lab_first_bicarbonate',
    'lab_first_lactate','lab_first_WBC x 1000','lab_first_Hgb','lab_first_platelets x 1000',
    'lab_first_sodium','lab_first_potassium','lab_first_BUN','lab_first_creatinine',
    'lab_first_calcium','lab_first_magnesium','lab_first_glucose','lab_first_troponin - T',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole'],
 'PMAP': ['gender','age','first_mGCS','flo_first_r_cpn_glasgow_coma_scale_score',
    'flo_first_bp_systolic','flo_first_bp_diastolic','flo_first_r_map',
    'flo_first_r_ed_pre-arrival_pulse_(heart_rate)','flo_first_r_sao2','flo_first_r_fio2',
    'flo_first_r_sofa_score','flo_first_r_bmi','flo_first_r_pao2','flo_first_r_paco2',
    'flo_first_r_resp_ph','lab_first_lactate','lab_first_troponin','lab_first_hemoglobin',
    'lab_first_platelet_count','lab_first_creatinine,whole_blood','lab_first_glucose,whole_blood',
    'lab_first_potassium,whole_blood','lab_first_sodium,whole_blood','lab_first_calcium,_serum',
    'lab_first_magnesium','lab_first_jhm_ip_qtc_ainterval_(sec)','asystole','pea','VF'],
 'MIMIC-IV': ['gender','age','bmi','first_mGCS',
    'chart_first_heart_rate','chart_first_o2_saturation_pulseoxymetry','chart_first_respiratory_rate',
    'chart_first_fio2_(ch)','chart_first_non_invasive_blood_pressure_systolic',
    'chart_first_non_invasive_blood_pressure_diastolic','chart_first_non_invasive_blood_pressure_mean',
    'chart_first_ph_(arterial)','chart_first_arterial_o2_pressure','chart_first_arterial_co2_pressure',
    'chart_first_lactic_acid','chart_first_wbc','chart_first_hemoglobin','lab_first_platelet_count',
    'chart_first_sodium_(serum)','lab_first_potassium_(serum)','chart_first_bun',
    'chart_first_creatinine_(serum)','chart_first_calcium_non-ionized','chart_first_magnesium',
    'chart_first_glucose_(serum)','lab_first_troponin-t','chart_first_qtc'],
 'HYPERION': ['J0_AGE','J0_SEX','J0_BMI','J0_PAS','J0_PAD','J0_PAM','J0_FC','J0_SPO2',
    'J0_GLASGOW','J0_MOTRICE','J0_RYTHM','J0_NOFLOW','J0_LOWFLOW','J0_IGSII',
    'BIO_LEUCO','BIO_HEMO','BIO_PLAQ','BIO_SODIUM','BIO_POTAS','BIO_UREE','BIO_CREAT',
    'BIO_CALCIUM','BIO_MAGNE','BIO_GLYCEMI','BIO_LACTAT','BIO_TROPO','BIO_PH','BIO_PAO2',
    'BIO_PACO2','BIO_BICARB'],
}

DML_NOTEBOOKS = {
    'eICU': os.path.join(REPO_ROOT, 'eICU', 'eICUAnalysisDML.ipynb'),
    'PMAP': os.path.join(REPO_ROOT, 'pmap', 'PMAPAnalysisDML.ipynb'),
    'MIMIC-IV': os.path.join(REPO_ROOT, 'mimiciv', 'MIMICAnalysisDML.ipynb'),
    'HYPERION': os.path.join(REPO_ROOT, 'hyperion', 'hyperionAnalysisDML.ipynb'),
}


def extract_dml_columns(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        nb = json.load(f)
    lists = []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        if 'columns' not in src:
            continue
        try:
            tree = ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == 'columns':
                        try:
                            val = ast.literal_eval(node.value)
                        except Exception:
                            continue
                        if isinstance(val, list) and all(isinstance(x, str) for x in val):
                            lists.append(val)
    return max(lists, key=len) if lists else None


DML_COLUMNS_REQUIRED = {'eICU', 'PMAP', 'MIMIC-IV'}
PARSED_DML_COLUMN_DATASETS = set()
for _name, _path in DML_NOTEBOOKS.items():
    _cols = extract_dml_columns(_path)
    if _cols:
        CURATED[_name] = _cols
        PARSED_DML_COLUMN_DATASETS.add(_name)
        print(f'Using {_name} full DML notebook columns: {len(_cols)} from {_path}')
    else:
        if _name == 'HYPERION':
            print(f'Using {_name} all numeric effective predictor columns; no explicit DML columns list found')
        else:
            print(f'No explicit DML columns list found for {_name}: {_path}')

_missing_required = sorted(DML_COLUMNS_REQUIRED - PARSED_DML_COLUMN_DATASETS)
if _missing_required:
    raise RuntimeError(
        'Missing parseable full DML `columns` list for required datasets: '
        + ', '.join(_missing_required)
    )

MIMIC_RHYTHM_PROXY_COLS = ['long_title_ventricular_fibrillation']
_before = len(CURATED['MIMIC-IV'])
CURATED['MIMIC-IV'] = [c for c in CURATED['MIMIC-IV'] if c not in MIMIC_RHYTHM_PROXY_COLS]
if len(CURATED['MIMIC-IV']) != _before:
    print('Excluded MIMIC-IV rhythm proxy columns:', MIMIC_RHYTHM_PROXY_COLS)


In [ ]:
SUPPLEMENT_FEATURES = {
  "eICU": [
    "gender",
    "age",
    "bmi",
    "nurse_first_Non-Invasive BP Systolic",
    "nurse_first_Non-Invasive BP Diastolic",
    "nurse_first_Non-Invasive BP Mean",
    "nurse_first_Heart Rate",
    "nurse_first_O2 Saturation",
    "nurse_first_Temperature (C)",
    "lab_first_Respiratory Rate",
    "lab_first_FiO2",
    "nurse_first_GCS Total",
    "nurse_first_Motor",
    "FirstGCS",
    "nurse_first_QTc",
    "lab_first_pH",
    "lab_first_paO2",
    "lab_first_paCO2",
    "lab_first_bicarbonate",
    "lab_first_lactate",
    "lab_first_WBC x 1000",
    "lab_first_Hgb",
    "lab_first_platelets x 1000",
    "lab_first_sodium",
    "lab_first_potassium",
    "lab_first_BUN",
    "lab_first_creatinine",
    "lab_first_calcium",
    "lab_first_magnesium",
    "lab_first_glucose",
    "lab_first_troponin - I",
    "lab_first_troponin - T",
    "diagnosis_initial rhythm: ventricular fibrillation",
    "diagnosis_ventricular fibrillation",
    "diagnosis_ventricular tachycardia",
    "diagnosis_initial rhythm: ventricular tachycardia",
    "diagnosis_initial rhythm: pulseless electrical activity",
    "diagnosis_initial rhythm: asystole"
  ],
  "MIMIC-IV": [
    "chart_seizure_duration_Status",
    "chart_cervical_collar_status_On",
    "med_sodium_bicarbonate",
    "input_sodium_bicarbonate_8.4%",
    "med_calcium_acetate",
    "input_calcium_gluconate",
    "med_miconazole_2%_cream",
    "chart_first_creatinine_(serum)",
    "chart_first_d-dimer",
    "output_first_d-dimer",
    "chart_first_fio2_(ch)",
    "chart_first_fio2_(ecmo)",
    "chart_first_glucose_(serum)",
    "chart_first_glucose_(whole_blood)",
    "chart_first_hemoglobin",
    "chart_problem_list_.H/O anemia, hemolytic",
    "med_calcium_acetate",
    "input_po_intake",
    "input_packed_red_blood_cells",
    "chart_first_hematocrit_(whole_blood_-_calc)",
    "chart_first_lipase",
    "output_first_lipase",
    "chart_first_magnesium",
    "input_magnesium_sulfate",
    "chart_first_peco2",
    "chart_first_etco2",
    "chart_first_peco2",
    "chart_first_svo2",
    "chart_first_gi_ph",
    "chart_first_ph_(venous)",
    "lab_first_platelet_count",
    "chart_first_absolute_count_-_eos",
    "input_po_intake",
    "lab_first_potassium_(serum)",
    "chart_first_total_protein",
    "output_first_total_protein",
    "input_heparin_sodium",
    "med_nitroprusside_sodium",
    "chart_lle_temp_Warm",
    "chart_rue_temp_Cold",
    "lab_first_prothrombin_time",
    "output_first_alt",
    "lab_first_troponin-t",
    "input_rocuronium",
    "chart_first_svo2",
    "chart_first_peco2",
    "lab_first_troponin-t",
    "input_tirofiban_(aggrastat)",
    "chart_first_arterial_blood_pressure_alarm_-_high",
    "chart_first_arterial_blood_pressure_mean",
    "chart_performance_level_Auto",
    "chart_performance_level_P2",
    "chart_performance_level_(r)_P6",
    "chart_performance_level_Auto",
    "chart_daily_wake_up_No, at goal RASS",
    "med_oxycodone_(immediate_release)_",
    "age",
    "chart_speech_Normal",
    "chart_lue_color_Normal",
    "chart_heart_rhythm_2nd AV M2 (Second degree AV Block - Mobitz 2)",
    "lab_first_ldl_measured",
    "lab_first_digoxin",
    "lab_first_ldl_measured",
    "lab_first_digoxin",
    "lab_first_ldl_measured",
    "chart_heart_rhythm_3rd AV (Complete Heart Block)",
    "chart_art_bubble_on_(ch)",
    "chart_urine_appearance_Cloudy",
    "chart_art_bubble_on_(ch)",
    "med_sarna_lotion",
    "chart_first_impella_catheter_position",
    "chart_lue_temp_Warm",
    "med_fentanyl_citrate",
    "chart_first_qtc",
    "output_first_qtc",
    "chart_temporary_ventricular_sens_Yes",
    "chart_temporary_ventricular_capture_Yes",
    "chart_problem_list_Ventricular tachycardia, sustained",
    "chart_first_vd/vt_ratio",
    "med_vasopressin",
    "chart_st_segment_monitoring_on",
    "chart_st_segment_monitoring_on",
    "med_sarna_lotion",
    "chart_problem_list_Tachycardia, Other",
    "chart_heart_rhythm_ST (Sinus Tachycardia)",
    "med_alteplase",
    "med_norepinephrine",
    "chart_heart_rhythm_V Paced",
    "chart_problem_list_Cardiac arrest",
    "chart_post-op_care_ncp_-_interventions_Assess surgical wounds and drains",
    "med_ursodiol",
    "chart_seizure_duration_Status",
    "med_sarna_lotion",
    "input_diltiazem",
    "chart_skin_condition_Diaphoretic",
    "chart_angio_site_#_1_R Femoral",
    "chart_first_ph_(venous)",
    "med_linezolid",
    "med_aspirin_ec",
    "med_sarna_lotion",
    "chart_impaired_fluid_balance_ncp_-_interventions_Monitoring electrolytes / renal function",
    "chart_first_hemoglobin",
    "chart_stroke_ncp_-_type_Hemorrhagic",
    "chart_first_intra_cranial_pressure",
    "chart_first_intra_cranial_pressure_#2",
    "chart_problem_list_Pulmonary edema",
    "chart_first_pulmonary_artery_pressure_mean",
    "med_tamsulosin",
    "med_insulin",
    "med_epinephrine_1:1000",
    "med_epinephrine",
    "med_epinephrine_1:1000",
    "med_epinephrine",
    "med_epinephrine_1:1000",
    "med_epinephrine",
    "age",
    "gender",
    "lab_first_platelet_count",
    "chart_pain_management_Repositioned",
    "med_vasopressin",
    "input_vasopressin",
    "med_sarna_lotion",
    "med_mannitol",
    "chart_first_arterial_blood_pressure_alarm_-_high",
    "chart_problem_list_Cerebrovascular disease, other",
    "chart_past_medical_history_COPD",
    "chart_cv_-_past_medical_history_CAD",
    "med_sodium_bicarbonate",
    "input_sodium_bicarbonate_8.4%",
    "med_sodium_bicarbonate",
    "input_sodium_bicarbonate_8.4%",
    "chart_first_bis_index_range",
    "chart_first_cardiac_index_(ci_nicom)",
    "chart_education_topic_Cardioversion",
    "chart_problem_list_Shock, cardiogenic",
    "chart_problem_list_Cardiac arrest",
    "chart_cardiac_assist_cannula_site_appear_WNL",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_gag_reflex_Intact",
    "chart_gag_reflex_Absent",
    "chart_incision_#1-_location_Tracheostomy",
    "chart_incision_#1-_location_Chin",
    "med_amiodarone",
    "input_amiodarone",
    "med_amiodarone",
    "input_amiodarone",
    "chart_corneal_reflex_left_Intact",
    "chart_corneal_reflex_left_Absent",
    "chart_daily_wake_up_deferred_Neuromusc Block",
    "chart_neuro_drain_#1_level_15 cm",
    "input_lidocaine",
    "input_amiodarone",
    "input_lidocaine",
    "med_midodrine",
    "chart_heart_rhythm_AF (Atrial Fibrillation)",
    "chart_problem_list_.H/O atrial fibrillation (Afib)",
    "chart_heart_rhythm_AF (Atrial Fibrillation)",
    "chart_problem_list_.H/O atrial fibrillation (Afib)",
    "med_nafcillin",
    "chart_first_heart_rate",
    "chart_first_fio2_(ch)",
    "chart_first_fio2_(ecmo)",
    "chart_first_respiratory_rate",
    "chart_first_respiratory_rate_(set)",
    "chart_20_g_infiltration_scale_Grade 0",
    "chart_20_g_infiltration_scale_Grade 2",
    "chart_goal_richmond-ras_scale_ No eye contact",
    "chart_20_g_infiltration_scale_Grade 0",
    "chart_cv_-_past_medical_history_Hypertension",
    "chart_problem_list_.H/O abdominal compartment syndrome / Intraabdominal Hypertension (IAH, ACS)",
    "hypothermia",
    "med_nephrocaps",
    "chart_pain_level_Unable to Score",
    "chart_pain_level_response_Unable to Score",
    "chart_circuit_configuration_(ch)_VA",
    "chart_par-circulation_BP +/- 20% of pre-anesthesthetic level",
    "chart_is_the_spokesperson_the_health_care_proxy",
    "chart_20_gauge_placed_in_the_field",
    "chart_problem_list_Cardiac arrest",
    "chart_ett_location_Oral-R",
    "chart_first_blood_flow_(ml/min)",
    "chart_first_flow_(ch)",
    "chart_pain_level_Unable to Score",
    "chart_pain_level_response_Unable to Score",
    "med_morphine_infusion_\u2013_comfort_care_guidelines",
    "med_norepinephrine",
    "chart_motor_l_leg_Localizes",
    "chart_motor_l_arm_Localizes",
    "chart_problem_list_Myocardial infarction, acute (AMI, STEMI, NSTEMI)",
    "underlying_cardiac_condition",
    "chart_first_flow_(ch)",
    "chart_first_flow_(ecmo)",
    "med_norepinephrine",
    "input_norepinephrine",
    "chart_first_heart_rate_alarm_-_low",
    "chart_first_heart_rate_alarm_-_high",
    "chart_svo2_sqi_1",
    "chart_first_svo2",
    "chart_eye_care",
    "chart_gcs_-_eye_opening_To Pain",
    "chart_first_lipase",
    "chart_first_manual_blood_pressure_diastolic_left",
    "input_diazepam_(valium)",
    "input_lorazepam_(ativan)",
    "chart_first_lipase",
    "chart_past_medical_history_COPD",
    "input_famotidine_(pepcid)",
    "lab_first_brain_natiuretic_peptide_(bnp)",
    "chart_first_daily_weight",
    "chart_first_feeding_weight",
    "chart_problem_list_Pulmonary edema",
    "chart_rul_lung_sounds_Diminished",
    "chart_pupil_size_right_Pinpoint",
    "chart_pupil_size_right_Fully Dilated",
    "chart_pupil_response_right_Non-reactive",
    "chart_pupil_response_left_Non-reactive",
    "chart_pupil_size_left_Pinpoint",
    "chart_pupil_size_left_5mm",
    "chart_pupil_response_left_Non-reactive",
    "chart_pupil_response_right_Non-reactive",
    "chart_gag_reflex_Intact",
    "chart_gag_reflex_Absent",
    "chart_gag_reflex_Intact",
    "chart_gag_reflex_Absent",
    "chart_gag_reflex_Absent",
    "chart_gag_reflex_Intact",
    "chart_heart_rhythm_SR (Sinus Rhythm)",
    "chart_heart_rhythm_ST (Sinus Tachycardia)",
    "chart_daily_wake_up_Yes, sedation adjusted or stopped",
    "chart_humidification_Active",
    "med_alteplase",
    "med_aspirin_ec",
    "chart_first_spo2_desat_limit",
    "output_first_spo2_desat_limit",
    "chart_problem_list_.H/O tobacco use, current",
    "chart_tobacco_use_history_Never used",
    "chart_first_height",
    "chart_first_height_(cm)",
    "chart_problem_list_Cardiac arrest",
    "chart_first_access_pressure",
    "chart_problem_list_Cardiac arrest",
    "chart_impaired_skin_cleanse_#6_Foam Cleanser",
    "chart_rue_temp_Warm",
    "chart_lle_temp_Cold",
    "chart_gcs_-_verbal_response_Oriented",
    "chart_gcs_-_verbal_response_No Response",
    "chart_first_spont_vt",
    "chart_first_vd/vt_ratio",
    "chart_emotional_/_physical_/_sexual_harm_by_partner_or_close_relation",
    "age",
    "chart_problem_list_Shock, cardiogenic",
    "chart_education_topic_Cardioversion",
    "med_sarna_lotion",
    "input_solution",
    "med_heparin",
    "chart_lle_temp_Cool",
    "chart_neuro_drain_#1_level_15 cm",
    "chart_neuro_drain_landmark_Tragus",
    "chart_circuit_configuration_(ch)_VA",
    "chart_problem_list_Renal failure, acute (Acute renal failure, ARF, AKI)",
    "chart_first_respiratory_rate",
    "chart_respiratory_effort_Normal",
    "chart_patient/family_informed_-----",
    "chart_back_care",
    "med_heparin",
    "chart_angio_site_#_1_R Femoral",
    "chart_angio_dressing_#_1_Bandaid",
    "chart_angio_dressing_#_1_Bandaid",
    "chart_angio_dressing_#_2_Bandaid",
    "chart_intra_aortic_balloon_pump_setting_1:2",
    "chart_intra_aortic_ballon_pump_setting_1:1",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "chart_alarms_on",
    "chart_slope_On",
    "med_nephrocaps",
    "chart_problem_list_Shock, cardiogenic",
    "chart_problem_list_Shock, cardiogenic",
    "chart_first_temporary_av_interval",
    "chart_angio_dressing_#_1_Bandaid",
    "chart_angio_dressing_#_2_Bandaid",
    "med_amiodarone",
    "chart_back_care",
    "hypothermia",
    "chart_impaired_tissue_perfusion_ncp_-_interventions_Therapeutic temperature management",
    "chart_stool_consistency_Loose",
    "lab_first_prothrombin_time",
    "chart_incision_#1-_treatment_Ace Wrap",
    "chart_incision_#1-_treatment_Xeroform",
    "hypothermia",
    "first_mGCS"
  ],
  "PMAP": [
    "gender",
    "age",
    "first_mGCS",
    "flo_first_r_cpn_glasgow_coma_scale_score",
    "flo_first_bp_systolic",
    "flo_first_bp_diastolic",
    "flo_first_r_map",
    "flo_first_temperature",
    "flo_first_r_ed_pre-arrival_pulse_(heart_rate)",
    "flo_first_r_fio2",
    "flo_first_r_spo2:fio2_covid-19_calculation",
    "flo_first_r_sofa_score",
    "flo_first_r_jhm_ip_sofa_cv_score",
    "flo_first_r_jhm_ip_sofa_cns_score",
    "flo_first_r_jhm_ip_sofa_coagulation_score",
    "flo_first_r_jhm_ip_sofa_liver_score",
    "flo_first_r_jhm_ip_sofa_renal_score",
    "flo_first_r_jhm_ip_pao2/fio2",
    "flo_first_r_jhm_ip_rt_vent_fio2_(%)",
    "flo_first_r_jhm_ip_rt_conv._vent._high_respiratory_rate",
    "flo_first_r_jhm_ip_rt_conv._vent._low_respiratory_rate",
    "flo_first_r_norepinephrine_volume",
    "flo_first_jhm_ip_4hr_urine_output_goal_(12ml/kg)_is_at_least",
    "flo_first_jhm_ip_4hr_urine_output_goal_(2ml/kg)_is_at_least",
    "flo_first_r_bmi",
    "lab_first_troponin",
    "lab_first_creatinine,whole_blood",
    "lab_first_glucose,whole_blood",
    "lab_first_potassium,whole_blood",
    "lab_first_sodium,whole_blood",
    "lab_first_hemoglobin,total,whole_blood",
    "lab_first_aptt",
    "hypothermia",
    "asystole",
    "pea",
    "cardiopulmonary arrest w/ resuscitation",
    "VF"
  ],
  "HYPERION": [
    "groupe",
    "J0_SEX",
    "J0_TAILLE",
    "J0_POIDS",
    "J0_BMI",
    "J0_AGE",
    "J0_PAS",
    "J0_PAD",
    "J0_PAM",
    "J0_FC",
    "J0_SPO2",
    "J0_GLASGOW",
    "J0_GLASGOW_CONTROLE",
    "J0_OCULAIRE",
    "J0_VERBALE",
    "J0_MOTRICE",
    "J0_PUPILG",
    "J0_PUPILG_REA",
    "J0_PUPILD",
    "J0_PUPILD_REA",
    "J0_CILIAIRE",
    "J0_CORNEEN",
    "J0_REFLEXVEST",
    "J0_REFLEXCEPH",
    "J0_REFLEXCARD",
    "J0_TEMP",
    "J0_IGSII",
    "J0_MCCABE",
    "J0_KNAUS",
    "J0_CHARLSON1",
    "J0_CHARLSON2",
    "J0_CHARLSON3",
    "J0_CHARLSON4",
    "J0_CHARLSON5",
    "J0_CHARLSON6",
    "J0_CHARLSON7",
    "J0_CHARLSON8",
    "J0_CHARLSON9",
    "J0_CHARLSON10",
    "J0_CHARLSON11",
    "J0_CHARLSON12",
    "J0_CHARLSON13",
    "J0_CHARLSON14",
    "V0_CHARLSON15",
    "V0_CHARLSON16",
    "V0_CHARLSON17",
    "V0_CHARLSON18",
    "V0_CHARLSON18B",
    "V0_CHARLSON19",
    "J0_CHARLSON",
    "J0_ATCD",
    "J0_CARDIO",
    "J0_NYHA",
    "J0_MYOCARD",
    "J0_ARTERIO",
    "J0_HTA",
    "J0_POUMON",
    "J0_IRC",
    "J0_HYPERCAP",
    "J0_O2",
    "J0_TABAC",
    "J0_RYTHM",
    "J0_CAUSE2_ACR",
    "J0_DSA",
    "J0_DSA_P",
    "J0_TEMOIN",
    "J0_TEMOIN_MASSE",
    "J0_LIEU_ACR",
    "J0_NOFLOW",
    "J0_LOWFLOW",
    "J0_ADRE",
    "J0_ADRE_DOS",
    "J0_CORDA",
    "J0_CORDA_DOS",
    "J0_BICARB",
    "J0_BICARB_DOS",
    "V0_REFROIDI",
    "V0_CHOC_AV",
    "V0_CHOC_AP",
    "V0_PLANCHE",
    "V0_THROMBO",
    "V0_CORO_ACR",
    "V0_ANGIO_ACR",
    "V0_ANGIO_YES",
    "V0_BALLON",
    "V0_ACR2",
    "J0_CURAR",
    "J0_SEDATIF",
    "J0_MORPHIN",
    "J0_COAGUL",
    "J0_AGREG",
    "J0_ANTIBIO",
    "J0_AMINE",
    "J0_NORA",
    "J0_ADRE2",
    "J0_DOBU",
    "J0_DOPA",
    "J0_PEP",
    "J0_FIO2",
    "J0_VT",
    "J0_FR",
    "BIO_LEUCO",
    "BIO_HEMO",
    "BIO_PLAQ",
    "BIO_TP",
    "BIO_DDIMERE",
    "BIO_SODIUM",
    "BIO_POTAS",
    "BIO_UREE",
    "BIO_CREAT",
    "BIO_CALCIUM",
    "BIO_MAGNE",
    "BIO_GLYCEMI",
    "BIO_PROTID",
    "BIO_LIPAS",
    "BIO_TROPO",
    "BIO_TEMP",
    "BIO_FIO2",
    "BIO_PH",
    "BIO_PAO2",
    "BIO_PACO2",
    "BIO_BICARB",
    "BIO_LACTAT",
    "BIO_TROPO2",
    "BIO_TROPO_CGT",
    "ECG",
    "ECG_QTC",
    "ECG_ANOMALI",
    "ECG_SUS_ST",
    "ECG_SOUS_ST",
    "ECG_BAVI",
    "ECG_BAVII",
    "ECG_BAVIII",
    "ECG_BBG",
    "ECG_BBD",
    "ECG_TACHICARD",
    "ECG_FIBRIL",
    "ECG_SALV_VENT",
    "ECG_FLUTER",
    "ECG_SALV_SUPRA",
    "SOFA_SC",
    "SOFA_RESPIR",
    "SOFA_CARDIO",
    "SOFA_COAG",
    "SOFA_NEURO",
    "SOFA_FOIE",
    "SOFA_RENAL",
    "EI_EI",
    "EI_HEMOSEVER",
    "EI_TRANSFUS",
    "EI_INTRACER",
    "EI_CHIR",
    "EI_EXTRARENAL",
    "EI_OAP",
    "EI_ECHO",
    "EI_DIURETIQ",
    "EI_CONVULS",
    "EI_ARYTHMI",
    "EI_ANTIEPILEPTIQ",
    "SOFA_SC1",
    "SH_DT_SORTI_H",
    "SEX"
  ]
}
SUPPLEMENT_FEATURE_SOURCE = "/Users/mraskin/Downloads/CCM submission (1)/Supplemental_Digital_Content_1_review.docx"
FEATURE_SOURCE_BY_DATASET = {
    name: ('full_dml_columns' if name in PARSED_DML_COLUMN_DATASETS else 'all_numeric_effective_predictors')
    for name in CURATED
}

if SUPPLEMENT_FEATURES:
    print('Using supplement eTables 16-19 feature lists from:', SUPPLEMENT_FEATURE_SOURCE)
    for _name, _cols in SUPPLEMENT_FEATURES.items():
        CURATED[_name] = _cols
        PARSED_DML_COLUMN_DATASETS.add(_name)
        FEATURE_SOURCE_BY_DATASET[_name] = 'supplement_etables_16_19'
        print(f'Using {_name} supplement feature list: {len(_cols)} variables')
else:
    print('No supplement feature lists embedded; using parsed DML columns/fallbacks.')


In [ ]:
UNSCORABLE = 'Unable to score due to medication'


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Missing {path}. Restore the predictor CSVs before executing the notebook.'
        )


def load_eicu():
    require_file(CSV['eICU'])
    df = pd.read_csv(CSV['eICU'])
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        if c in df.columns:
            df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df['gender'] = (df['gender'] == 'Male').astype(int)
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['nurse_first_Motor'] != 6) & ~df['Hypothermia'].isna()].copy()
    return df.rename(columns={
        'Hypothermia': 'TTM',
        'DeathAtDischarge': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def _load_epic_style(name):
    require_file(CSV[name])
    df = pd.read_csv(CSV[name])
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    df = df[(df['first_mGCS'] != 6) & ~df['hypothermia'].isna()].copy()
    return df.rename(columns={
        'hypothermia': 'TTM',
        'death_at_disch': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def load_pmap():
    return _load_epic_style('PMAP')


def load_mimic():
    return _load_epic_style('MIMIC-IV')


def load_hyperion():
    require_file(CSV['HYPERION'])
    df = pd.read_csv(CSV['HYPERION'])
    df = df[df['groupe'] != 2].copy()
    df['TTM'] = (df['groupe'] == 1).astype(int)
    return df.rename(columns={
        'hospital_mortality': 'mortality',
        'CPC12': 'neuro_favorable',
    })


LOADERS = {
    'eICU': load_eicu,
    'PMAP': load_pmap,
    'MIMIC-IV': load_mimic,
    'HYPERION': load_hyperion,
}


def effective_feature_source(df, dataset):
    drop = {'TTM', 'mortality', 'neuro_favorable'}
    if dataset == 'eICU':
        temp_cols = [c for c in df.columns if 'emp' in c]
        drop.update(temp_cols)
        drop.update([
            'FirstGCS', 'LastMGCSTime', 'FirstMGCSTime', 'LastMGCSPositive', 'LastMGCS',
            'apacheadmissiondx', 'hospitaladmittime24', 'FirstGCSTime', 'LastGCSTime',
            'LastGCS', 'hospitaldischargestatus', 'LastGCS15', 'hospitaladmitsource',
            'DeathAtDischarge', 'Hypothermia', 'patientunitstayid', 'hypothermia_time',
        ])
    elif dataset == 'PMAP':
        temp_cols = [c for c in df.columns if ('emp' in c and c[:3] == 'dx_')]
        drop.update(temp_cols)
        drop.update([
            'first_mGCS', 'last_mGCS_time', 'first_mGCS_time', 'LastMGCSPositive',
            'last_mGCS', 'death_at_disch', 'hypothermia',
        ])
    elif dataset == 'MIMIC-IV':
        temp_cols = [c for c in df.columns if ('emp' in c or c[:3] == 'dx_')]
        drop.update(temp_cols)
        drop.update([
            'last_mGCS_time', 'first_mGCS_time', 'LastMGCSPositive',
            'last_mGCS', 'death_at_disch', 'hypothermia',
            'long_title_ventricular_fibrillation',
        ])
    elif dataset == 'HYPERION':
        # HYPERION: use all available numeric predictor columns. Exclude only treatment,
        # identifiers, endpoints, and clearly post-outcome/follow-up variables.
        drop.update([
            'CPC_SC3', 'CPC12', 'SUBJID', 'BARTHEL_SC', 'SOFA_SC7', 'DS_DC',
            'DAYS_ALIVE_30', 'hospital_mortality', 'groupe',
        ])

    X = df.drop(columns=[c for c in drop if c in df.columns], errors='ignore')
    X = X.select_dtypes(exclude=['object'])
    if dataset != 'HYPERION':
        binary_cols = X.columns[X.nunique(dropna=True) == 2]
        low_cols = X[binary_cols].sum(numeric_only=True)
        low_cols = low_cols[low_cols < 15].index.tolist()
        X = X.drop(columns=low_cols, errors='ignore')
    return X


def load_analysis_frame(dataset, outcome):
    df = LOADERS[dataset]()
    feature_source = effective_feature_source(df, dataset)
    if dataset in PARSED_DML_COLUMN_DATASETS:
        # Match the DML notebooks' columns_filter behavior exactly: use every parsed DML
        # feature column that survives the dataset loader/effective-feature exclusions.
        cols = sorted(set(c for c in CURATED[dataset] if c in feature_source.columns))
    elif dataset == 'HYPERION':
        cols = list(feature_source.columns)
    else:
        raise ValueError(
            f'{dataset} must use its full DML notebook columns, but no parseable `columns` '
            f'list was found in {DML_NOTEBOOKS[dataset]}.'
        )
    if dataset in PARSED_DML_COLUMN_DATASETS:
        missing = [c for c in CURATED[dataset] if c not in df.columns]
        excluded_by_dml = [c for c in CURATED[dataset] if c in df.columns and c not in feature_source.columns]
    else:
        missing = []
        excluded_by_dml = []
    X = feature_source[cols].apply(pd.to_numeric, errors='coerce')
    T = pd.to_numeric(df['TTM'], errors='coerce')
    y = pd.to_numeric(df[outcome], errors='coerce')
    keep = T.notna() & y.notna()
    out = X.loc[keep].reset_index(drop=True)
    out['TTM'] = T.loc[keep].astype(int).reset_index(drop=True)
    out[outcome] = y.loc[keep].astype(int).reset_index(drop=True)
    meta = {
        'dataset': dataset,
        'outcome': outcome,
        'n': int(keep.sum()),
        'n_ttm': int(T.loc[keep].sum()),
        'n_features': len(cols),
        'feature_source': FEATURE_SOURCE_BY_DATASET.get(dataset, 'all_numeric_effective_predictors'),
        'dml_columns_requested': len(CURATED[dataset]) if dataset in PARSED_DML_COLUMN_DATASETS else np.nan,
        'missing_curated_features': missing,
        'dml_columns_excluded_by_loader': excluded_by_dml,
    }
    return out, cols, meta


cohort_rows = []
for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        try:
            _, _, meta = load_analysis_frame(ds['name'], outcome_col)
            meta['outcome_name'] = outcome_name
            cohort_rows.append(meta)
        except Exception as e:
            cohort_rows.append({
                'dataset': ds['name'],
                'outcome': outcome_col,
                'outcome_name': outcome_name,
                'status': f'missing: {e}',
            })

cohort_table = pd.DataFrame(cohort_rows)
cohort_table.to_csv(os.path.join(OUTDIR, 'cohort_table.csv'), index=False)
display(cohort_table)


In [ ]:
def make_stratify(y, T):
    return pd.Series(y).astype(str) + '_' + pd.Series(T).astype(str)


def _make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def getDefaultPreprocessor(aNumericalColumns, aBinaryColumns):
    transformers = []
    numerical = list(aNumericalColumns)
    binary = list(aBinaryColumns)
    if numerical:
        transformers.append(('num', StandardScaler(), numerical))
    if binary:
        transformers.append(('bin', 'passthrough', binary))
    return ColumnTransformer(
        transformers=transformers,
        remainder=_make_one_hot_encoder(),
        verbose_feature_names_out=False,
    )


def getDefaultPipelineSteps(X_train):
    myNumericalColumns = X_train.columns[(X_train.nunique() > 5) & (X_train.dtypes != object)]
    myBinaryColumns = X_train.columns[X_train.nunique() == 2]
    myPreprocessor = getDefaultPreprocessor(aNumericalColumns=myNumericalColumns, aBinaryColumns=myBinaryColumns)
    return [('preprocessor', myPreprocessor), ('imputer', KNNImputer(n_neighbors=10))]


def transform_fold(df, feature_cols, train_idx, val_idx, outcome_col):
    X_train_raw = df.loc[train_idx, feature_cols]
    X_val_raw = df.loc[val_idx, feature_cols]
    prep = Pipeline(getDefaultPipelineSteps(X_train_raw))
    prep.fit(X_train_raw)
    X_train = pd.DataFrame(
        prep.transform(X_train_raw),
        columns=prep.get_feature_names_out(),
        index=X_train_raw.index,
    ).reset_index(drop=True)
    X_val = pd.DataFrame(
        prep.transform(X_val_raw),
        columns=prep.get_feature_names_out(),
        index=X_val_raw.index,
    ).reset_index(drop=True)
    y_train = df.loc[train_idx, outcome_col].astype(int).to_numpy()
    y_val = df.loc[val_idx, outcome_col].astype(int).to_numpy()
    T_train = df.loc[train_idx, 'TTM'].astype(int).to_numpy()
    T_val = df.loc[val_idx, 'TTM'].astype(int).to_numpy()
    return X_train, X_val, y_train, y_val, T_train, T_val, prep


def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    y = np.asarray(y)
    p = np.asarray(p)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    if not aucs:
        return np.nan, np.nan
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def calibration_slope(y, p):
    p = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
    logit_p = np.log(p / (1 - p))
    X = sm.add_constant(logit_p)
    try:
        m = sm.Logit(y, X).fit(disp=False)
        return float(m.params[1])
    except Exception:
        return np.nan


def lrt_cate_interaction(y, T, cate):
    cate = np.asarray(cate, float)
    if np.ptp(cate) < 1e-12:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': 'CATE constant'}
    d = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float), 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    y = np.asarray(y, float)
    try:
        m0 = sm.Logit(y, d[['const', 'T', 'cate']]).fit(disp=False)
        m1 = sm.Logit(y, d[['const', 'T', 'cate', 'tx']]).fit(disp=False)
        lr = 2 * (m1.llf - m0.llf)
        return {'lr_stat': float(lr), 'p': float(chi2.sf(lr, 1)), 'note': ''}
    except Exception as e:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': str(e)}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    T = np.asarray(T, float)
    y = np.asarray(y, float)
    cate = np.asarray(cate, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    if np.ptp(cate) < 1e-12:
        return {'wald_p': np.nan, 'note': 'CATE constant'}
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    d = pd.DataFrame({'const': 1.0, 'T': T, 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    try:
        m = sm.GLM(y, d[['const', 'T', 'cate', 'tx']], family=sm.families.Binomial(),
                   freq_weights=w).fit(cov_type='HC1')
        return {'wald_p': float(m.pvalues['tx']), 'note': ''}
    except Exception as e:
        return {'wald_p': np.nan, 'note': str(e)}


def run_gates(cate, y, T, ps=None, observational=False, title='', save_prefix=None):
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float), 'cate': np.asarray(cate, float)})
    if observational and ps is not None:
        ps = np.clip(np.asarray(ps, float), *PS_CLIP)
        d['w'] = np.where(d['T'] == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    else:
        d['w'] = 1.0
    d['group'] = pd.qcut(d['cate'].rank(method='first'), q=N_GATES_GROUPS, labels=False) + 1
    rows = []
    for g, sub in d.groupby('group'):
        m = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = np.asarray(m.conf_int())
        rows.append({
            'group': int(g),
            'n': len(sub),
            'mean_cate': sub['cate'].mean(),
            'gate': m.params.iloc[1],
            'ci_low': ci[1, 0],
            'ci_high': ci[1, 1],
        })
    gdf = pd.DataFrame(rows)
    gd = pd.get_dummies(d['group'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])
    if save_prefix:
        fig, ax = plt.subplots(figsize=(5.8, 4.0))
        ax.errorbar(gdf['group'], gdf['gate'],
                    yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                    fmt='o-', capsize=4)
        ax.axhline(0, color='0.4', ls='--', lw=1)
        ax.set_xlabel('Predicted CATE quintile')
        ax.set_ylabel('Group average treatment effect')
        ax.set_title(f'{title}\nF p={f_p:.3f}; rho={rho:.2f} (p={rho_p:.3f})')
        fig.tight_layout()
        fig.savefig(os.path.join(OUTDIR, f'{save_prefix}_gates.png'), dpi=180)
        plt.close(fig)
    return gdf, {'f_stat': float(f_stat), 'f_p': float(f_p), 'spearman_rho': float(rho), 'spearman_p': float(rho_p)}


In [ ]:
def slearner_design(X, T):
    Xd = X.copy()
    Xd['TTM'] = np.asarray(T, float)
    return Xd


def predict_slearner_cate(model, X):
    x0 = X.copy()
    x1 = X.copy()
    x0['TTM'] = 0.0
    x1['TTM'] = 1.0
    p0 = model.predict_proba(x0)[:, 1]
    p1 = model.predict_proba(x1)[:, 1]
    return p1 - p0, p0, p1


def fit_xgb_slearner(X_train, T_train, y_train):
    model = XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1)
    grid = {
        'n_estimators': [25, 50, 100, 200],
        'max_depth': [2, 5, 10],
        'learning_rate': [0.03, 0.1],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    gs = GridSearchCV(model, grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(slearner_design(X_train, T_train), y_train)
    return gs.best_estimator_, gs.best_params_


def fit_lasso_slearner(X_train, T_train, y_train):
    Xd = slearner_design(X_train, T_train)
    base_cols = list(X_train.columns)
    for c in base_cols:
        Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
    model = LogisticRegressionCV(
        penalty='l1',
        solver='saga',
        Cs=[1e-3, 1e-2, 1e-1, 1.0],
        cv=5,
        scoring='neg_log_loss',
        max_iter=20000,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(Xd, y_train)
    return model, {'chosen_C': float(model.C_[0]), 'interaction_cols': [f'{c}_x_TTM' for c in base_cols]}


def predict_lasso_slearner(model, X, interaction_cols):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        for c in X.columns:
            Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
        return Xd
    p0 = model.predict_proba(build(0))[:, 1]
    p1 = model.predict_proba(build(1))[:, 1]
    return p1 - p0, p0, p1


def fit_neural_slearner(X_train, T_train, y_train, dataset):
    nn_cfg = {
        'eICU':     {'layers': [64, 64, 32],       'dropout': 0.5, 'lr': 1e-3, 'epochs': 20},
        'PMAP':     {'layers': [64, 64, 64, 64, 32],'dropout': 0.2, 'lr': 1e-4, 'epochs': 50},
        'MIMIC-IV': {'layers': [64, 64, 64, 32],   'dropout': 0.2, 'lr': 1e-3, 'epochs': 60},
        'HYPERION': {'layers': [64],               'dropout': 0.5, 'lr': 1e-3, 'epochs': 15},
    }[dataset]
    Xd_df = slearner_design(X_train, T_train)
    if NEURAL_BACKEND != 'keras' or keras is None:
        # Cluster-safe CPU neural net. This avoids TensorFlow CUDA/PTX failures while preserving
        # the S-learner neural-network sensitivity row.
        model = MLPClassifier(
            hidden_layer_sizes=tuple(nn_cfg['layers']),
            activation='relu',
            solver='adam',
            learning_rate_init=nn_cfg['lr'],
            alpha=1e-4,
            batch_size=32,
            max_iter=max(100, nn_cfg['epochs'] * 5),
            early_stopping=True,
            n_iter_no_change=10,
            random_state=SEED,
        )
        model.fit(Xd_df, np.asarray(y_train).astype(int))
        out_cfg = dict(nn_cfg)
        out_cfg['backend'] = 'sklearn_mlp'
        return model, out_cfg

    tf.keras.utils.set_random_seed(SEED)
    Xd = Xd_df.to_numpy(dtype='float32')
    y = np.asarray(y_train).astype('float32')
    inp = keras.Input(shape=(Xd.shape[1],))
    z = inp
    for width in nn_cfg['layers']:
        z = layers.Dense(width, activation='relu')(z)
        z = layers.Dropout(nn_cfg['dropout'])(z)
    out = layers.Dense(1, activation='sigmoid')(z)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=nn_cfg['lr']),
                  loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')],
                  jit_compile=False)
    if USE_TF_GPU:
        model.fit(Xd, y, batch_size=32, epochs=nn_cfg['epochs'], validation_split=0.2, verbose=0)
    else:
        with tf.device('/CPU:0'):
            model.fit(Xd, y, batch_size=32, epochs=nn_cfg['epochs'], validation_split=0.2, verbose=0)
    out_cfg = dict(nn_cfg)
    out_cfg['backend'] = 'keras'
    return model, out_cfg


def predict_neural_slearner(model, X):
    def build_df(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd
    if hasattr(model, 'predict_proba'):
        p0 = model.predict_proba(build_df(0))[:, 1]
        p1 = model.predict_proba(build_df(1))[:, 1]
    else:
        p0 = model.predict(build_df(0).to_numpy(dtype='float32'), verbose=0).ravel()
        p1 = model.predict(build_df(1).to_numpy(dtype='float32'), verbose=0).ravel()
    return p1 - p0, p0, p1


def fit_bart_slearner(X_train, T_train, y_train, dataset):
    if pm is None or pmb is None:
        raise RuntimeError('PyMC/pymc_bart is not available')
    Xd = slearner_design(X_train, T_train).to_numpy(dtype='float64')
    y = np.asarray(y_train).astype(int)
    m_trees = {'eICU': 30, 'PMAP': 30, 'MIMIC-IV': 50, 'HYPERION': 30}[dataset]
    with pm.Model() as model:
        X_shared = pm.Data('X_shared', Xd)
        u = pmb.BART('u', X=X_shared, Y=y, m=m_trees)
        p = pm.Deterministic('p', pm.math.sigmoid(u))
        pm.Bernoulli('y', p=p, observed=y)
        trace = pm.sample(
            draws=BART_DRAWS[dataset],
            tune=BART_TUNE,
            chains=BART_CHAINS,
            cores=BART_CORES,
            random_seed=SEED,
            progressbar=True,
        )
    return model, trace, {'m_trees': m_trees, 'draws': BART_DRAWS[dataset], 'tune': BART_TUNE}


def predict_bart_slearner(model, trace, X):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd.to_numpy(dtype='float64')
    preds = []
    with model:
        for val in [0, 1]:
            pm.set_data({'X_shared': build(val)})
            ppc = pm.sample_posterior_predictive(trace, var_names=['p'], random_seed=SEED, progressbar=False)
            arr = np.asarray(ppc.posterior_predictive['p'])
            preds.append(arr.reshape(-1, arr.shape[-1]).mean(axis=0))
    p0, p1 = preds
    return p1 - p0, p0, p1


def fit_causal_forest(X_train, T_train, y_train, dataset):
    if CausalForestDML is None:
        raise RuntimeError('econml is not available')
    # Keep HYPERION settings aligned with the submitted DML notebook; observational settings
    # match the current revision sensitivity notebooks.
    if dataset == 'HYPERION':
        model_y = XGBClassifier(max_depth=25, n_estimators=100, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=10, n_estimators=20, random_state=SEED, eval_metric='logloss')
    elif dataset == 'PMAP':
        model_y = XGBClassifier(max_depth=5, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    else:
        model_y = XGBClassifier(max_depth=3, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        discrete_outcome=True,
        random_state=SEED,
        n_jobs=-1,
    )
    cf.fit(y_train, T_train, X=X_train, cache_values=True)
    return cf, {'model_y': str(model_y), 'model_t': str(model_t)}


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


In [ ]:
RESULTS = []
GATES_ROWS = []
IMPORTANCE_ROWS = []


def add_result(row):
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(os.path.join(OUTDIR, 'all_model_results_partial.csv'), index=False)


def evaluate_oof_slearner(dataset, outcome_name, model_name, cate, p_obs, y, T, extra=None):
    lrt = lrt_cate_interaction(y, T, cate)
    auc = roc_auc_score(y, p_obs) if len(np.unique(y)) == 2 else np.nan
    ci_low, ci_high = bootstrap_auc_ci(y, p_obs)
    row = {
        'dataset': dataset,
        'outcome': outcome_name,
        'model': model_name,
        'n_eval': len(y),
        'evaluation_method': EVAL_METHOD,
        'cv_splits': N_SPLITS if EVAL_METHOD == 'crossfit' else 1,
        'test_size': TEST_SIZE if EVAL_METHOD == 'split' else np.nan,
        'lr_p': lrt['p'],
        'lr_note': lrt['note'],
        'auroc': auc,
        'auroc_ci_low': ci_low,
        'auroc_ci_high': ci_high,
        'brier': brier_score_loss(y, p_obs),
        'calibration_slope': calibration_slope(y, p_obs),
        'mean_cate': float(np.mean(cate)),
        'cate_p05': float(np.percentile(cate, 5)),
        'cate_p50': float(np.percentile(cate, 50)),
        'cate_p95': float(np.percentile(cate, 95)),
    }
    if extra:
        row.update(extra)
    add_result(row)


def run_dataset_outcome(dataset, outcome_name, outcome_col, observational):
    print(f'\n===== {dataset} / {outcome_name} =====')
    try:
        df, feature_cols, meta = load_analysis_frame(dataset, outcome_col)
    except Exception as e:
        add_result({
            'dataset': dataset,
            'outcome': outcome_name,
            'model': 'all',
            'status': f'skipped before modeling: {e}',
        })
        print('Skipped:', e)
        return
    y_all = df[outcome_col].astype(int).to_numpy()
    T_all = df['TTM'].astype(int).to_numpy()
    n = len(df)
    if EVAL_METHOD == 'split':
        print(f'n={n}; stratified {1 - TEST_SIZE:.0%}/{TEST_SIZE:.0%} train/test split; features={len(feature_cols)}')
    else:
        print(f'n={n}; {N_SPLITS}-fold stratified CV; features={len(feature_cols)}')

    oof = {}
    for model_name in ['S-learner XGBoost', 'S-learner L1 logistic', 'S-learner neural network',
                       'S-learner BART', 'CausalForestDML/R-learner']:
        oof[model_name] = {
            'cate': np.full(n, np.nan),
            'p_obs': np.full(n, np.nan),
            'lo': np.full(n, np.nan),
            'hi': np.full(n, np.nan),
            'ps': np.full(n, np.nan),
            'extras': [],
        }

    strat = make_stratify(y_all, T_all)
    if EVAL_METHOD == 'split':
        train_idx, val_idx = train_test_split(
            np.arange(n), test_size=TEST_SIZE, random_state=SEED, stratify=strat
        )
        partitions = [(1, np.asarray(train_idx), np.asarray(val_idx))]
        partition_label = 'split'
    else:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        partitions = [(fold, train_idx, val_idx)
                      for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(n), strat), start=1)]
        partition_label = f'{N_SPLITS}-fold CV'

    for fold, train_idx, val_idx in partitions:
        if EVAL_METHOD == 'split':
            print(f'  split: train={len(train_idx)}, test={len(val_idx)}')
        else:
            print(f'  fold {fold}/{N_SPLITS}: train={len(train_idx)}, validation={len(val_idx)}')
        X_train, X_val, y_train, y_val, T_train, T_val, prep = transform_fold(
            df, feature_cols, train_idx, val_idx, outcome_col
        )

        if RUN_XGB_SLEARNER:
            try:
                model, params = fit_xgb_slearner(X_train, T_train, y_train)
                cate, p0, p1 = predict_slearner_cate(model, X_val)
                oof['S-learner XGBoost']['cate'][val_idx] = cate
                oof['S-learner XGBoost']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner XGBoost']['extras'].append(params)
                if hasattr(model, 'feature_importances_'):
                    for col, val in zip(slearner_design(X_train, T_train).columns, model.feature_importances_):
                        IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name, 'fold': fold,
                                                'model': 'S-learner XGBoost', 'feature': col, 'importance': float(val)})
            except Exception as e:
                print(f'    XGBoost failed fold {fold}:', e)

        if RUN_LASSO_SLEARNER:
            try:
                model, info = fit_lasso_slearner(X_train, T_train, y_train)
                cate, p0, p1 = predict_lasso_slearner(model, X_val, info['interaction_cols'])
                oof['S-learner L1 logistic']['cate'][val_idx] = cate
                oof['S-learner L1 logistic']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                coefs = pd.Series(model.coef_[0], index=slearner_design(X_train, T_train).columns.tolist() + info['interaction_cols'])
                oof['S-learner L1 logistic']['extras'].append({
                    'chosen_C': info['chosen_C'],
                    'nonzero_interactions': int((coefs[info['interaction_cols']] != 0).sum()),
                    'n_interactions': len(info['interaction_cols']),
                })
            except Exception as e:
                print(f'    L1 logistic failed fold {fold}:', e)

        if RUN_NEURAL_SLEARNER:
            try:
                model, cfg = fit_neural_slearner(X_train, T_train, y_train, dataset)
                cate, p0, p1 = predict_neural_slearner(model, X_val)
                oof['S-learner neural network']['cate'][val_idx] = cate
                oof['S-learner neural network']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner neural network']['extras'].append(cfg)
            except Exception as e:
                print(f'    Neural failed fold {fold}:', e)

        if RUN_BART_SLEARNER:
            try:
                model, trace, cfg = fit_bart_slearner(X_train, T_train, y_train, dataset)
                cate, p0, p1 = predict_bart_slearner(model, trace, X_val)
                oof['S-learner BART']['cate'][val_idx] = cate
                oof['S-learner BART']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner BART']['extras'].append(cfg)
            except Exception as e:
                print(f'    BART failed fold {fold}:', e)

        if RUN_CAUSAL_FOREST:
            try:
                cf, cfg = fit_causal_forest(X_train, T_train, y_train, dataset)
                cate = np.ravel(cf.effect(X_val))
                lo, hi = cf.effect_interval(X_val, alpha=0.05)
                oof['CausalForestDML/R-learner']['cate'][val_idx] = cate
                oof['CausalForestDML/R-learner']['lo'][val_idx] = np.ravel(lo)
                oof['CausalForestDML/R-learner']['hi'][val_idx] = np.ravel(hi)
                oof['CausalForestDML/R-learner']['ps'][val_idx] = (
                    cf_propensity(cf, X_val) if observational else np.repeat(T_train.mean(), len(val_idx))
                )
                oof['CausalForestDML/R-learner']['extras'].append(cfg)
                if hasattr(cf, 'feature_importances_'):
                    for col, val in zip(X_train.columns, cf.feature_importances_):
                        IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name, 'fold': fold,
                                                'model': 'CausalForestDML/R-learner',
                                                'feature': col, 'importance': float(val)})
            except Exception as e:
                print(f'    CausalForest failed fold {fold}:', e)

    for model_name, d in oof.items():
        ran = ~np.isnan(d['cate'])
        if not ran.any():
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': model_name,
                        'status': 'failed: no folds produced predictions'})
            continue
        if model_name == 'CausalForestDML/R-learner':
            lrt = lrt_cate_interaction(y_all[ran], T_all[ran], d['cate'][ran])
            ps = d['ps'][ran]
            ipw = ipw_interaction_test(y_all[ran], T_all[ran], d['cate'][ran], ps) if observational else {'wald_p': np.nan, 'note': 'randomized'}
            gates, gates_stats = run_gates(
                d['cate'][ran], y_all[ran], T_all[ran], ps=ps, observational=observational,
                title=f'{dataset} {outcome_name} CausalForestDML ({partition_label})',
                save_prefix=f'{dataset}_{outcome_name}_causal_forest_{EVAL_METHOD}'.replace(' ', '_').replace('/', '_')
            )
            gates.insert(0, 'dataset', dataset)
            gates.insert(1, 'outcome', outcome_name)
            gates.insert(2, 'model', model_name)
            GATES_ROWS.extend(gates.to_dict('records'))
            row = {
                'dataset': dataset,
                'outcome': outcome_name,
                'model': model_name,
                'n_eval': int(ran.sum()),
                'evaluation_method': EVAL_METHOD,
                'cv_splits': N_SPLITS if EVAL_METHOD == 'crossfit' else 1,
                'test_size': TEST_SIZE if EVAL_METHOD == 'split' else np.nan,
                'lr_p': lrt['p'],
                'lr_note': lrt['note'],
                'ipw_interaction_p': ipw['wald_p'],
                'gates_f_p': gates_stats['f_p'],
                'gates_spearman_rho': gates_stats['spearman_rho'],
                'gates_spearman_p': gates_stats['spearman_p'],
                'pct_cate_ci_cross_zero': float(np.mean((d['lo'][ran] < 0) & (d['hi'][ran] > 0))),
                'mean_cate': float(np.mean(d['cate'][ran])),
                'cate_p05': float(np.percentile(d['cate'][ran], 5)),
                'cate_p50': float(np.percentile(d['cate'][ran], 50)),
                'cate_p95': float(np.percentile(d['cate'][ran], 95)),
            }
            add_result(row)
        else:
            extra = {}
            if model_name == 'S-learner L1 logistic' and d['extras']:
                extra = {
                    'chosen_C': float(np.median([x['chosen_C'] for x in d['extras']])),
                    'nonzero_interactions': float(np.median([x['nonzero_interactions'] for x in d['extras']])),
                    'n_interactions': int(np.median([x['n_interactions'] for x in d['extras']])),
                }
            evaluate_oof_slearner(dataset, outcome_name, model_name,
                                  d['cate'][ran], d['p_obs'][ran], y_all[ran], T_all[ran], extra)


for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        run_dataset_outcome(ds['name'], outcome_name, outcome_col, ds['observational'])

results = pd.DataFrame(RESULTS)
gates_table = pd.DataFrame(GATES_ROWS)
importance_table = pd.DataFrame(IMPORTANCE_ROWS)

results.to_csv(os.path.join(OUTDIR, 'all_model_results.csv'), index=False)
gates_table.to_csv(os.path.join(OUTDIR, 'gates_results.csv'), index=False)
importance_table.to_csv(os.path.join(OUTDIR, 'variable_importance.csv'), index=False)

display(results)


In [ ]:
results = pd.DataFrame(RESULTS)

for col in ['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier',
            'calibration_slope', 'nonzero_interactions', 'n_interactions', 'chosen_C']:
    if col not in results.columns:
        results[col] = np.nan

table2 = results.pivot_table(
    index=['dataset', 'outcome', 'model'],
    values=['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier', 'calibration_slope'],
    aggfunc='first',
).reset_index()
table2.to_csv(os.path.join(OUTDIR, 'table2_model_performance_and_lr.csv'), index=False)
display(table2.round(4))

cf_summary = results[results['model'].eq('CausalForestDML/R-learner')].copy()
cf_summary.to_csv(os.path.join(OUTDIR, 'causal_forest_hte_summary.csv'), index=False)
display(cf_summary.round(4))

lasso_summary = results[results['model'].eq('S-learner L1 logistic')][
    ['dataset', 'outcome', 'nonzero_interactions', 'n_interactions', 'chosen_C', 'lr_p']
].copy()
lasso_summary.to_csv(os.path.join(OUTDIR, 'lasso_slearner_summary.csv'), index=False)
display(lasso_summary)


In [ ]:
def fmt_p(x):
    if pd.isna(x):
        return 'NA'
    return '<0.001' if x < 0.001 else f'{x:.3f}'


print('--- Train/test and preprocessing ---')
if EVAL_METHOD == 'split':
    print(f'Within each dataset, patients were evaluated using a stratified {(1 - TEST_SIZE):.0%}/{TEST_SIZE:.0%} train/test split, stratified jointly by treatment and outcome. Standard scaling, KNN imputation (k=10), and all model tuning were fit on the training partition only; metrics and HTE tests use held-out test predictions.')
else:
    print(f'Within each dataset, patients were evaluated with {N_SPLITS}-fold stratified cross-validation, stratified jointly by treatment and outcome. Standard scaling, KNN imputation (k=10), and all model tuning were fit on each fold training partition only; metrics and HTE tests use out-of-fold predictions.')

print('\n--- CausalForestDML/R-learner HTE summary ---')
for _, r in cf_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: LR p={fmt_p(r['lr_p'])}; "
          f"GATES F p={fmt_p(r['gates_f_p'])}; "
          f"CATE 95% CIs crossing zero={r['pct_cate_ci_cross_zero']:.1%}; "
          f"IPW interaction p={fmt_p(r.get('ipw_interaction_p', np.nan))}.")

print('\n--- L1 S-learner regularization summary ---')
for _, r in lasso_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: {int(r['nonzero_interactions'])}/{int(r['n_interactions'])} treatment interactions survived the L1 penalty; LR p={fmt_p(r['lr_p'])}.")
